# HAM10000 / ISIC-Style EfficientNet-B0 Training

Research-only. Not for diagnosis, treatment decisions, or clinical deployment. Doctor review required.

This notebook is separate from the DermaCon-IN OPD model. Use it for dermoscopy-style lesion labels such as `mel`, `nv`, `bcc`, `bkl`, `akiec`, `df`, `vasc`.

## 1. Runtime

In Colab: `Runtime` -> `Change runtime type` -> choose GPU.

In [ ]:
!nvidia-smi

## 2. Mount Drive And Paths

Place HAM10000 files under:

```text
+MyDrive/derm-opd-triage-ham10000/data/raw/HAM10000_metadata.csv
+MyDrive/derm-opd-triage-ham10000/data/raw/HAM10000_images_part_1/*.jpg
+MyDrive/derm-opd-triage-ham10000/data/raw/HAM10000_images_part_2/*.jpg
+```

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_PROJECT = '/content/drive/MyDrive/derm-opd-triage-ham10000'
REPO_URL = 'https://github.com/Abhigyan-Shekhar/skin-lesion-detect-model.git'
REPO_DIR = '/content/skin-lesion-detect-model'

Path(DRIVE_PROJECT).mkdir(parents=True, exist_ok=True)
(Path(DRIVE_PROJECT) / 'data' / 'raw').mkdir(parents=True, exist_ok=True)
(Path(DRIVE_PROJECT) / 'outputs_ham10000').mkdir(parents=True, exist_ok=True)
print('Drive project folder ready:', DRIVE_PROJECT)

## 3. Clone Repo And Install

In [ ]:
import subprocess
from pathlib import Path

def run_checked(cmd, *, cwd=None):
    print('+', ' '.join(cmd))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr)
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return result

if not Path(REPO_DIR).exists():
    run_checked(['git', 'clone', REPO_URL, REPO_DIR])
else:
    run_checked(['git', 'pull'], cwd=REPO_DIR)

%cd {REPO_DIR}
!pip install -q -r requirements.txt

## 4. Link HAM10000 Data

Download HAM10000 separately, then upload/unzip into the Drive layout above. Kaggle download usually requires your Kaggle account/API token, so this notebook assumes files are already in Drive.

In [ ]:
from pathlib import Path
import shutil

drive_raw = Path(DRIVE_PROJECT) / 'data' / 'raw'
repo_ham = Path(REPO_DIR) / 'data' / 'ham10000'
repo_raw = repo_ham / 'raw'
repo_splits = repo_ham / 'splits'
repo_ham.mkdir(parents=True, exist_ok=True)
repo_splits.mkdir(parents=True, exist_ok=True)

metadata = drive_raw / 'HAM10000_metadata.csv'
part1 = drive_raw / 'HAM10000_images_part_1'
part2 = drive_raw / 'HAM10000_images_part_2'

if not metadata.exists():
    raise FileNotFoundError(f'Missing metadata: {metadata}')
if not part1.exists() or not part2.exists():
    raise FileNotFoundError('Missing HAM10000_images_part_1 or HAM10000_images_part_2 in Drive raw folder')

if repo_raw.exists() or repo_raw.is_symlink():
    if repo_raw.is_symlink():
        repo_raw.unlink()
    else:
        shutil.rmtree(repo_raw)
repo_raw.symlink_to(drive_raw, target_is_directory=True)
print('Linked', repo_raw, '->', drive_raw)
!find data/ham10000/raw -maxdepth 2 -type f | head -20

## 5. Prepare Splits

In [ ]:
!python src/prepare_ham10000.py \
  --metadata data/ham10000/raw/HAM10000_metadata.csv \
  --image_dirs data/ham10000/raw/HAM10000_images_part_1 data/ham10000/raw/HAM10000_images_part_2 \
  --output_dir data/ham10000/splits

!cat data/ham10000/splits/split_summary.json

## 6. Train HAM10000 EfficientNet-B0

In [ ]:
!python src/train.py --config configs/efficientnet_b0_ham10000.yaml

## 7. Evaluate

In [ ]:
!python src/evaluate.py \
  --checkpoint outputs_ham10000/checkpoints/best.pt \
  --split data/ham10000/splits/test.csv \
  --output_dir outputs_ham10000

!cat outputs_ham10000/metrics/metrics.json
!ls -R outputs_ham10000 | sed -n '1,140p'

## 8. Save Artifacts Back To Drive

In [ ]:
from pathlib import Path
import shutil

drive_outputs = Path(DRIVE_PROJECT) / 'outputs_ham10000'
drive_outputs.mkdir(parents=True, exist_ok=True)

for folder in ['checkpoints', 'metrics', 'plots', 'predictions']:
    src = Path(REPO_DIR) / 'outputs_ham10000' / folder
    dst = drive_outputs / folder
    if src.exists():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print('Copied', src, '->', dst)

!find {DRIVE_PROJECT}/outputs_ham10000 -maxdepth 3 -type f | sort

## 9. Inference Test

In [ ]:
from google.colab import files
uploaded = files.upload()

image_path = next(iter(uploaded.keys()))
!python src/inference.py --checkpoint outputs_ham10000/checkpoints/best.pt --image "$image_path" --top_k 5